# Demo - Shape Deformation for Bandsaw Cutting

This is a minimal setup to generate similar shapes for bandsaw cutting as in the paper (Sec. 4.3).

This notebook requires more dependencies than the `dvx-python` module. Take sure to install the dependencies:

````powershell
python -m venv .env
.env/Scripts/Activate.ps1 # depends on your os
pip install -r demos/requirements.txt
````

In [ ]:
import math
import numpy as np
import torch
import tqdm
import dvx.torch as dvx
import arap # as-rigid-as-possible energy in torch
import util # utility functions to keep the notebook clean
import tempfile, os
import polyscope as ps

torch.manual_seed(0);

## Setup parameters

In [ ]:
### configuration
mesh_path = "./meshes/hand.obj" # path to input mesh
d = 128                          # voxel resolution
max_steps = 1500                 # maximum number of optimization steps
step = 0.001                     # learning rate
alpha = 0.005                    # weight of arap loss
eps=5e-4                         # converged if max vertex change is below eps

# load and normalize mesh
V, F = util.load_obj(mesh_path)
V -= V.mean(0)
V /= torch.linalg.vector_norm(V, dim=1).max() * 1.25 # we leave some space for deformation

## Loss definition and helper functions

In [ ]:
def loss_fn(U: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
  # deformation loss
  loss_deform = arap.ARAP_energy(U, V, F)

  # cut loss
  q = dvx.voxelize(d, U, F) # call the differentiable voxelizer
  q_cut = cut_from_volume(q)
  loss_bandsaw = 1./(d**3) * (q_cut - q).pow(2).sum() # additional volume produced by bandsaw

  loss = loss_bandsaw + alpha * loss_deform
  return loss, loss_bandsaw, loss_deform

# helper
def mellowmax(q: torch.Tensor, dim: int, w: float = 1000, keepdim=False) -> torch.Tensor:
  """ The mellowmax operation on tensor q along dimension dim with weight w.
  This is a smooth way for projecting a 3D volume to a 2D slice along dim.
  w -> 0 gives the mean and w -> inf gives max along dim.
  """
  m = q.max(dim, keepdim=True).values
  lse = torch.exp((q - m)*w).sum(dim=dim, keepdim=keepdim).log()
  result = m + (lse - math.log(q.size(dim)))/w
  return result

def cut_from_volume(p: torch.Tensor) -> torch.Tensor:
  """ This returns the cut out mesh when cutting an object with the silhouettes of 3d tensor p.
  """
  p_x = mellowmax(p, 0, keepdim=True) # 2D silhouette when looking along x-axis
  p_y = mellowmax(p, 1, keepdim=True)
  p_z = mellowmax(p, 2, keepdim=True)
  p_cut = p_x * p_y * p_z # condition for a voxel to not be removed
  return p_cut

# Optimization Loop

In [ ]:
U = V.clone() # deformed vertices
U.requires_grad_(True)
optim = torch.optim.Adam([U], lr=step)
_, init_loss_bandsaw, _ = loss_fn(U)
print(f"Initial bandsaw loss: {init_loss_bandsaw:.5f}).")

# optimize
itr = tqdm.trange(max_steps, desc=f"Loss: -")
try:
  for t in itr:
    # bookkeeping
    loss, loss_bandsaw, loss_deform = loss_fn(U)
    itr.set_description(f"Loss: {loss:.5f}")

    U_prev = U.detach().clone()
    optim.zero_grad()
    loss.backward()
    optim.step()

    converged = torch.linalg.vector_norm(U.detach()-U_prev, dim=1).max().item() < eps
    if converged:
      break
except KeyboardInterrupt:
  print(f"Skipping {max_steps - t} iterations.")

improved = 1 - loss_bandsaw/init_loss_bandsaw
print(f"Final loss: {loss:.5f} (Bandsaw: {loss_bandsaw:.5f}, Deform {loss_deform:.5f}). Relative cut improvement: {improved:.4f}.")

## Show results

In [ ]:
def load_rdbu_colormap(name="rdbu_half"):
  """ Register the gray-to-red half of matplotlib's 'RdBu' (ColorBrewer) as a polyscope colormap. """
  colors = ['f7f7f7', 'fddbc7', 'f4a582', 'd6604d', 'b2182b', '67001f'] # center -> red end of RdBu
  c = np.array([[int(h[i:i+2], 16) for i in (0, 2, 4)] for h in colors], dtype=np.float64)
  t, k = np.linspace(0, 1, 256), np.linspace(0, 1, len(c))
  row = np.stack([np.interp(t, k, c[:, j]) for j in range(3)], 1).round().astype(np.uint8)
  path = os.path.join(tempfile.gettempdir(), f"{name}.ppm")
  with open(path, 'wb') as f:
    f.write(b'P6 256 1 255\n' + row.tobytes())
  # polyscope keeps colormaps for the lifetime of the kernel and warns when re-loading one,
  # so silence that warning (there is no API to check whether a colormap already exists)
  ps.set_verbosity(0)
  ps.set_display_message_popups(False)
  ps.load_color_map(name, path)
  ps.set_verbosity(2)                # polyscope defaults
  ps.set_display_message_popups(True)

ps.init()
ps.remove_all_structures()             # re-running the cell would otherwise keep old structures and transforms
ps.set_ground_plane_mode("shadow_only") # hide the ground plane but keep the shadows
load_rdbu_colormap()

with torch.no_grad():
  voxel_saw = dvx.voxelize(d, U, F)
  voxel_orig = dvx.voxelize(d, V, F)
  voxel_saw_cut = cut_from_volume(voxel_saw)
  voxel_orig_cut = cut_from_volume(voxel_orig)

  disp = torch.linalg.vector_norm(U - V, dim=1).numpy() # vertex displacement magnitude
disp_max = disp.max()

cell_width = 3*[2./d]
gray = [0.969, 0.969, 0.969] # center of RdBu, appears gray when shaded

def register_cut(name, voxel_cut, voxel_ref, offset):
  occupied = voxel_cut > 0.5
  inside = voxel_ref > 0.5                       # voxels of the voxelized surface
  idx = occupied.nonzero(as_tuple=False).numpy()
  extra = (occupied & ~inside)[occupied].numpy() # additional voxels produced by the bandsaw
  values = extra.astype(np.float64)              # 0 -> gray, 1 -> red

  origin = (np.array([-1., -1., -1.]) + offset).tolist()
  grid = ps.register_sparse_volume_grid(name, origin, cell_width, idx[:, ::-1]) # Polyscope expects [x,y,z], the voxelization is [z,y,x]
  grid.add_scalar_quantity("surface / additional", values, defined_on='cells', cmap='rdbu_half', vminmax=(0., 1.), enabled=True)
  return grid

# 2x2 layout: columns = original | optimized, rows = voxel grids | meshes
col = 1.1 # half the distance between the columns (x)
row = 2.2 # distance between the rows (z); use -row to put the meshes on the other side

ps_grid_orig = register_cut("Cut shape (original)", voxel_orig_cut, voxel_orig, [-col, 0., 0.])
ps_grid_saw = register_cut("Cut shape (optimized)", voxel_saw_cut, voxel_saw, [ col, 0., 0.])

ps_mesh_orig = ps.register_surface_mesh("Mesh (original)", V.numpy(), F.numpy(), color=gray,
                                        edge_width=.4, edge_color=[0., 0., 0.])
ps_mesh_saw = ps.register_surface_mesh("Mesh (optimized)", U.detach().numpy(), F.numpy(),
                                       edge_width=.4, edge_color=[0., 0., 0.])
ps_mesh_saw.add_scalar_quantity("displacement", disp, defined_on='vertices', cmap='rdbu_half', vminmax=(0., disp_max), enabled=True)
ps_mesh_orig.set_position([-col, 0., row])
ps_mesh_saw.set_position([ col, 0., row])

ps.show()